# Mount the drive

In [ ]:
#Connecting this collab notebook to my account.
from google.colab import drive
drive.mount('/content/drive')

# Load a saved model
Please change the path value below

In [ ]:
import tensorflow as tf
#path of the saved model
loaded_model = tf.keras.models.load_model('/content/drive/MyDrive/project sample/Models/best_model_mobilenetv2.h5')

# Test Images Path

In [ ]:
# Provide the test path to the folder
src_folder = "/content/drive/MyDrive/project sample/Test"

# Dictionary of the labels predicted by the model


In [ ]:
import os

#predicted labels
dict_labels = [file for file in os.listdir(src_folder) if os.path.isdir(os.path.join(src_folder, file))]
dict_labels.sort()
dict_labels = dict(enumerate(dict_labels))
print(dict_labels)

# Create a list of the images in the test and train folders

In [ ]:
import os
directory_contents = []
# Find all subfolders in the source folder src_folder
for index in range(0,len(dict_labels)):
  directory_contents.append(dict_labels[index])
print(directory_contents)

# List the files in each subfolder
list_files = []
# Find all the files
for folder_name in directory_contents:
  files_list = os.listdir(os.path.join(src_folder, folder_name))
  list_files.append(files_list)
print(list_files)

# Create the predictions and label lists

In [ ]:
from keras.applications.resnet import preprocess_input
from tensorflow.keras.preprocessing import image
import numpy as np
# A counter to increment after processing each categoty
category_count = 0
# label list
label = []
# Prediction list
predictions = []
y_score = []
for categories in list_files:
  # Create the labels while iterating over each category
  label_temp = np.ones((len(categories))).astype(int)*category_count
  print(label_temp)
  # Add it to existing labels
  label.extend(label_temp)
  for file_name in categories:
    # test image file
    img_path = src_folder + '/' + directory_contents[category_count] + '/' + file_name
    img = image.load_img(img_path, target_size=(224,224))
    img_array = image.img_to_array(img)
    img_batch = np.expand_dims(img_array, axis=0)
    img_preprocessed = preprocess_input(img_batch)
    prediction = loaded_model.predict(img_preprocessed)
    y_score.append(prediction)
    # Save the index of maximum probability
    predictions.append(np.argmax(prediction))
  category_count += 1


### ROC Curve

In [ ]:
probs = np.array([i[0] for i in y_score])
#ROC curve
from sklearn.metrics import roc_curve
import matplotlib.pyplot as plt
# 0s are negative and 1s are positive
fpr, tpr, _ = roc_curve(label, probs[:,1])

plt.figure(figsize = (8,5))
plt.plot(fpr,tpr, label = "ROC_CURVE", lw = 2, color='darkorange')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Receiver operating characteristic (ROC_CURVE)")
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.00])
plt.legend(loc = 'lower right')
plt.show()

# Confusion Matrix

In [ ]:
'''
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
display_labels = sorted(directory_contents)
plt.figure(figsize = (10,10))
sns.heatmap(confusion_matrix(label,predictions),
            annot = True,
            fmt = 'g',
            cmap = "Blues",
            xticklabels=display_labels,
            yticklabels = display_labels,
            annot_kws={
                'fontsize': 8,
                'fontweight': 'bold',
                'fontfamily': 'serif'
            })
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")
#ConfusionMatrixDisplay.from_predictions(label, predictions, display_labels=display_labels, cmap="binary")
plt.show()'''

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.metrics import confusion_matrix

display_labels = sorted(directory_contents)

def plot_confusion_matrix(y_true, y_pred, labels):
    # Compute normalized confusion matrix
    cm = confusion_matrix(y_true, y_pred, normalize='true')

    # Convert to percentages
    cm_percentage = cm * 100

    # Create annotations for confusion matrix values with the percentage sign
    annotations = np.array([[f'{val:.2f}%' for val in row] for row in cm_percentage])

    # Plot the confusion matrix
    plt.figure(figsize=(10, 10))
    sns.heatmap(cm_percentage, annot=annotations, fmt='', cmap="Blues",
                xticklabels=labels, yticklabels=labels)

    plt.title("Normalized Confusion Matrix")
    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.show()

In [ ]:
#plot_confusion_matrix(y_validation, predictions, display_labels)

plot_confusion_matrix(label, predictions, display_labels)

# Classification Report

In [ ]:
#classification reports
from sklearn.metrics import classification_report

print(classification_report(label,predictions))

# Precision and Recall Score
- This is for each category with labels.
- CSV is also generated for later use.

In [ ]:
#precision score and recall score
from sklearn.metrics import precision_score, recall_score
import pandas as pd

precision = precision_score(label, predictions, average=None)
recall = recall_score(label, predictions, average=None)

dict1 = {}
dict2 = {}

for num in range(len(recall)):
    dict1[dict_labels[num]] = precision[num]
    dict2[dict_labels[num]] = recall[num]

# Create dataframe directly
df_metrics = pd.DataFrame([dict1, dict2])

df_metrics['metrics'] = ['precision', 'recall']
df_metrics.set_index('metrics', inplace=True)

df_metrics.to_csv("precision_recall.csv", index=True)
print("CSV Saved")